# Inspect SLEAP pose, heading direction and trajectory

Quick read-only checks of the pose pipeline for one session:
`pose.SleapRawTracking` → `PoseQC` → `PositionPixel` / `PositionCM` / `HeadingDirection` (foragingABC) and `ovc_behav.EarHeading` (ear-axis heading).

Run on HPC with the repo `.venv` kernel. Populate the downstream tables first:
`python scripts/pipeline/populate_pose.py <experiment> --downstream`.

In [ ]:
import json
import os
from pathlib import Path

# Run from the repo root so datajoint.json is picked up
root = Path.cwd()
while not (root / "datajoint.json").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
print("Working directory:", root)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from aeon.dj_pipeline import acquisition
from foragingabc.dj_pipeline import pose
from foragingabc.dj_pipeline.utils.sleap import cm_convert
from src.dj_pipeline import ovc_behav

EXPERIMENT = "ProjectAeonOVC-2026-09-14_001"
SLEAP_METHOD = "ovc_top_260921_legacy"
CAMERA = "CameraTop"
CHUNK_INDEX = None   # None = chunk with the fewest unlabeled frames
WINDOW_S = 20        # length of the trajectory window (s)
ARROW_EVERY = 10     # draw a heading arrow every N frames
ARROW_PX = 40        # arrow length in raw-image pixels

restriction = {"experiment_name": EXPERIMENT, "sleap_method": SLEAP_METHOD}

## Progress per table

In [ ]:
tables = {
    "SleapRawTracking": pose.SleapRawTracking,
    "PoseQC": pose.PoseQC,
    "PositionPixel": pose.PositionPixel,
    "PositionCM": pose.PositionCM,
    "HeadingDirection": pose.HeadingDirection,
    "EarHeading": ovc_behav.EarHeading,
}
n_total = len(pose.SleapRawTracking.key_source & restriction)
for name, table in tables.items():
    print(f"{name:18s} {len(table & restriction):4d} / {n_total}")

## QC overview and chunk selection

In [ ]:
qc = pd.DataFrame((pose.PoseQC & restriction).to_dicts()).sort_values("chunk_start").reset_index(drop=True)
display(qc[["chunk_start", "frac_unlabeled", "frac_low_confidence", "n_jumps", "n_frozen"]])

fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(len(qc)), qc["frac_unlabeled"], color="0.4")
ax.set_xticks(range(0, len(qc), 6), [str(t)[5:16] for t in qc["chunk_start"][::6]], rotation=45, ha="right")
ax.set_ylabel("fraction unlabeled")
ax.set_title("Frames without a detected animal, per chunk")
plt.show()

idx = CHUNK_INDEX if CHUNK_INDEX is not None else int(qc["frac_unlabeled"].idxmin())
key = {k: qc.loc[idx, k] for k in pose.PoseQC.primary_key}
print(f"Chunk {idx}: {key['chunk_start']}  (frac_unlabeled = {qc.loc[idx, 'frac_unlabeled']:.3f})")
kp_missing = qc.loc[idx, "keypoint_missing_frac"]
kp_missing = json.loads(kp_missing) if isinstance(kp_missing, str) else kp_missing  # MariaDB returns json as text
print("Keypoint missing fraction:", {k: round(v, 3) for k, v in kp_missing.items()})

## Load the chunk

All headings use the image convention: 0° = up in the camera image, clockwise, y pointing down.

In [ ]:
ds_pose = (pose.SleapRawTracking & key).fetch1("pose_data")
ds_px = (pose.PositionPixel & key).fetch1("position_data")
ds_cm = (pose.PositionCM & key).fetch1("position_data")
ds_ear = (ovc_behav.EarHeading & key).fetch1("heading_data")
ds_abc = (pose.HeadingDirection & key).fetch1("heading_direction_data")

times = ds_pose.coords["time"].values
fps = float(ds_pose.attrs.get("fps") or 50.0)

def centroid(ds):
    p = ds["position"].sel(keypoint="centroid").isel(individual=0)
    return p.sel(space="x").values, p.sel(space="y").values

u_px, v_px = centroid(ds_px)
x_cm, y_cm = centroid(ds_cm)
ear_hd = ds_ear["heading_direction"].isel(individual=0).values
ear_conf = ds_ear["confidence"].isel(individual=0).values
abc_hd = ds_abc["heading_direction"].isel(individual=0).values

print(f"{len(times)} frames, {fps} fps, video: {ds_pose.attrs.get('video_path')}")
print(f"ear_sign_flip_frac: {ds_ear.attrs.get('ear_sign_flip_frac')}  "
      "(small = ear labels consistent; near 1 = image mirrored vs. labels)")
print(f"frames with ear heading: {np.isfinite(ear_hd).mean():.1%}, with foragingABC heading: {np.isfinite(abc_hd).mean():.1%}")

## Heading over the whole chunk: ear axis vs foragingABC (spine → head)

In [ ]:
step = 5  # plot every 5th frame to keep it light
t_min = (times - times[0]) / np.timedelta64(1, "m")

fig, axes = plt.subplots(3, 1, figsize=(16, 8), sharex=True, constrained_layout=True)
axes[0].scatter(t_min[::step], ear_hd[::step], s=1, c=ear_conf[::step], cmap="viridis", vmin=0, vmax=1, rasterized=True)
axes[0].set_ylabel("ear heading (°)")
axes[1].scatter(t_min[::step], abc_hd[::step], s=1, color="tab:orange", rasterized=True)
axes[1].set_ylabel("foragingABC heading (°)")
axes[2].plot(t_min[::step], ear_conf[::step], lw=0.5, color="0.3")
axes[2].set_ylabel("ear confidence")
axes[2].set_xlabel("minutes into chunk")
for ax in axes[:2]:
    ax.set_ylim(0, 360)
    ax.set_yticks([0, 90, 180, 270, 360])
fig.suptitle(f"{EXPERIMENT} | {key['chunk_start']}")
plt.show()

diff = ((ear_hd - abc_hd + 180) % 360) - 180
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(diff[np.isfinite(diff)], bins=90, range=(-180, 180), color="0.4")
ax.set_xlabel("ear − foragingABC heading (°)")
ax.set_ylabel("frames")
plt.show()
print(f"median |difference|: {np.nanmedian(np.abs(diff)):.1f}°")

## Pick an active window (highest median speed)

In [ ]:
speed = np.hypot(np.diff(x_cm, prepend=np.nan), np.diff(y_cm, prepend=np.nan)) * fps
speed = pd.Series(speed).rolling(int(0.5 * fps), center=True, min_periods=1).median()
win = int(WINDOW_S * fps)
window_speed = speed.rolling(win, min_periods=int(0.8 * win)).median()
i_end = int(np.nanargmax(window_speed.values))
i0, i1 = max(i_end - win + 1, 0), i_end + 1
sel = slice(i0, i1)
print(f"Window {times[i0]} → {times[i_end]}  (median speed {window_speed.iloc[i_end]:.1f} cm/s, "
      f"chunk p99 {np.nanpercentile(speed, 99):.1f} cm/s)")

## Trajectory with heading arrows (arena cm)

Arrows are built in raw-pixel space (centroid + `ARROW_PX` along the heading), then both ends are mapped to cm
with `cm_convert.map_pixel_to_cm`, so they use the same calibration as `PositionCM`.

In [ ]:
def arrows_cm(hd_deg, frames):
    th = np.deg2rad(hd_deg[frames])
    bu, bv = u_px[frames], v_px[frames]
    tu, tv = bu + ARROW_PX * np.sin(th), bv - ARROW_PX * np.cos(th)
    bx, by = cm_convert.map_pixel_to_cm(CAMERA, bu, bv)
    tx, ty = cm_convert.map_pixel_to_cm(CAMERA, tu, tv)
    return bx, by, tx - bx, ty - by

frames = np.arange(i0, i1, ARROW_EVERY)
img, extent = (pose.PositionCM & key).background_image(time=times[i0])

kp = list(ds_pose.coords["keypoint"].values)
raw = ds_pose["position"].isel(individual=0)  # (time, space, keypoint)
ear_cm = {name: cm_convert.map_pixel_to_cm(CAMERA, raw.sel(keypoint=name, space="x").values[sel],
                                          raw.sel(keypoint=name, space="y").values[sel])
          for name in ("left_ear", "right_ear")}

fig, axes = plt.subplots(1, 2, figsize=(18, 9), constrained_layout=True)
for ax, zoom in zip(axes, (False, True)):
    ax.imshow(img[..., ::-1], extent=extent)
    sc = ax.scatter(x_cm[sel], y_cm[sel], c=(times[sel] - times[i0]) / np.timedelta64(1, "s"), s=3, cmap="plasma")
    for hd, color, label in ((ear_hd, "cyan", "ear heading"), (abc_hd, "orange", "foragingABC heading")):
        bx, by, dx, dy = arrows_cm(hd, frames)
        ax.quiver(bx, by, dx, dy, angles="xy", scale_units="xy", scale=1, color=color, width=0.003, label=label)
    if zoom:
        ax.scatter(*ear_cm["left_ear"], s=2, color="lime", label="left ear")
        ax.scatter(*ear_cm["right_ear"], s=2, color="red", label="right ear")
        pad = 15
        ax.set_xlim(np.nanmin(x_cm[sel]) - pad, np.nanmax(x_cm[sel]) + pad)
        ax.set_ylim(np.nanmin(y_cm[sel]) - pad, np.nanmax(y_cm[sel]) + pad)
        ax.legend(loc="upper right", markerscale=4)
    ax.set_xlabel("x (cm)")
    ax.set_ylabel("y (cm)")
axes[0].set_title(f"{WINDOW_S} s window from {str(times[i0])[:19]}")
axes[1].set_title("zoom")
fig.colorbar(sc, ax=axes, label="time in window (s)", shrink=0.6)
plt.show()

## Single frames in raw pixels

The cyan arrow (ear heading, from the ear midpoint) should point out of the snout.

In [ ]:
raw_dir = acquisition.Experiment.get_data_directory(key, directory_type="raw")
video_path = Path(raw_dir) / ds_pose.attrs["video_path"]
cap = cv2.VideoCapture(str(video_path))

frame_ids = np.linspace(i0, i1 - 1, 4).astype(int)
colors = plt.cm.tab10(np.arange(len(kp)))
fig, axes = plt.subplots(1, len(frame_ids), figsize=(20, 5.5), constrained_layout=True)
for ax, f in zip(axes, frame_ids):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(f))
    ok, frame = cap.read()
    if not ok:
        ax.set_title(f"frame {f}: read failed")
        continue
    ax.imshow(frame[..., ::-1])
    pts = raw.isel(time=f).values  # (space, keypoint)
    for k, name in enumerate(kp):
        ax.scatter(pts[0, k], pts[1, k], s=25, color=colors[k], label=name)
    L, R = pts[:, kp.index("left_ear")], pts[:, kp.index("right_ear")]
    mid = (L + R) / 2
    for hd, color, base in ((ear_hd, "cyan", mid), (abc_hd, "orange", np.array([u_px[f], v_px[f]]))):
        if np.isfinite(hd[f]) and np.isfinite(base).all():
            th = np.deg2rad(hd[f])
            ax.annotate("", xy=(base[0] + ARROW_PX * np.sin(th), base[1] - ARROW_PX * np.cos(th)), xytext=base,
                        arrowprops=dict(arrowstyle="->", color=color, lw=2))
    c = np.array([u_px[f], v_px[f]])
    if np.isfinite(c).all():
        ax.set_xlim(c[0] - 150, c[0] + 150)
        ax.set_ylim(c[1] + 150, c[1] - 150)
    ax.set_title(f"{str(times[f])[11:23]}  ear {ear_hd[f]:.0f}°  abc {abc_hd[f]:.0f}°  conf {ear_conf[f]:.2f}")
    ax.axis("off")
cap.release()
axes[0].legend(loc="lower left", fontsize=8)
plt.show()